# Goal

The goal of this project is to perform exploratory data analysis on the IEEE-CIS Fraud Detection dataset and identify patterns that may help separate fraudulent from non-fraudulent transactions.

This is a medium-to-large, highly imbalanced tabular dataset with hundreds of anonymized features. The project focuses on:

- memory-aware data handling;
- missingness analysis;
- fraud-rate analysis by interpretable feature groups;
- identifying useful preprocessing decisions for later modeling.

This notebook is focused on EDA. Modeling, experiment tracking, SHAP analysis, and deployment will be handled in separate notebooks/modules.

Run this notebook in Google Colab with Kaggle credentials in Colab secrets. The exported feature list is learned from the earlier 80% of transactions only.

In [1]:
# Pin the XGBoost version used for the Colab feature-selection artifacts.
import importlib.metadata
import subprocess
import sys
try:
    installed_xgb = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    installed_xgb = None
if installed_xgb != "3.4.1":
    if "xgboost" in sys.modules:
        raise RuntimeError("Restart the Colab runtime before installing XGBoost 3.4.1")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.4.1"])
import xgboost as xgb
assert xgb.__version__ == "3.4.1"
print("Feature-selection XGBoost:", xgb.__version__)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.express as px

try:
    from google.colab import userdata
    key = userdata.get("KAGGLE_API_TOKEN")
    if key:
        os.environ["KAGGLE_API_TOKEN"] = key.strip()
except ImportError:
    pass

Feature-selection XGBoost: 3.4.1


In [2]:
# Install the Kaggle CLI only when this Colab runtime needs it.
import importlib.metadata
import subprocess
import sys

try:
    kaggle_version = importlib.metadata.version("kaggle")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_version = importlib.metadata.version("kaggle")
print("Kaggle CLI:", kaggle_version)

Kaggle CLI: 2.0.2


# Helper functions

In [3]:
import warnings
from contextlib import nullcontext

def reduce_memory(df: pd.DataFrame, verbose: bool = True, ignore_warnings: bool = False, inplace: bool = True) -> pd.DataFrame:
    """
    Downcast numeric columns to reduce DataFrame memory usage.
    """
    if not inplace:
        df = df.copy()

    start_mem = df.memory_usage(deep=True).sum() / 1e6

    warning_context = warnings.catch_warnings() if ignore_warnings else nullcontext()

    with warning_context:
        if ignore_warnings:
            warnings.filterwarnings(
                "ignore",
                category=RuntimeWarning,
                message="overflow encountered in cast",
            )

        for col in df.columns:
            series = df[col]

            if pd.api.types.is_integer_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = series.astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = series.astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = series.astype(np.int32)
                else:
                    df[col] = series.astype(np.int64)

            elif pd.api.types.is_float_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = series.astype(np.float32)
                else:
                    df[col] = series.astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1e6

    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem if start_mem else 0
        print(f"Memory: {start_mem:.2f} MB → {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df

# Data load

In [4]:
import subprocess

archive = Path("/content/ieee-fraud-detection.zip")
required_csvs = [Path("/content/train_transaction.csv"), Path("/content/train_identity.csv")]
if not all(path.exists() for path in required_csvs):
    if not archive.exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", "/content"],
            check=True,
        )
    subprocess.run(["unzip", "-qo", str(archive), "-d", "/content"], check=True)
if not all(path.exists() for path in required_csvs):
    raise FileNotFoundError("Kaggle training CSVs are missing from /content")
print("Training CSVs ready in /content")


Training CSVs ready in /content


In [5]:
BASE_PATH = Path("/content")
TRAIN_TRANSACTION = BASE_PATH / "train_transaction.csv"
TRAIN_IDENTITY = BASE_PATH / "train_identity.csv"

# ID ranking needs only the transaction key, time, label, and identity data.
train_transaction = reduce_memory(pd.read_csv(
    TRAIN_TRANSACTION, usecols=["TransactionID", "TransactionDT", "isFraud"]
), True)
train_identity = reduce_memory(pd.read_csv(TRAIN_IDENTITY), True)
print("Loaded", len(train_transaction), "transactions and", len(train_identity), "identity rows")


Memory: 14.17 MB → 5.31 MB (62.5% reduction)
Memory: 150.10 MB → 136.25 MB (9.2% reduction)
Loaded 590540 transactions and 144233 identity rows


In [6]:
import gc

# Inputs were downcast as they were read.
identity_columns = train_identity.columns.tolist()
train_df_merged = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")
del train_transaction, train_identity
gc.collect()


90

Only the transaction key, time, label, and identity columns are loaded.
Numeric columns are downcast once before the merge. The latest 20% of
transactions is reserved for evaluation and excluded from ID ranking.


# ID features

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

id_features_all = [col for col in train_df_merged.columns if col.startswith("id_")]
time_order = train_df_merged["TransactionDT"].sort_values(kind="stable").index
train_period_idx = time_order[:int(len(time_order) * 0.8)]
final_holdout_idx = time_order[int(len(time_order) * 0.8):]
y = train_df_merged.loc[train_period_idx, "isFraud"]
inner_cutoff = int(len(train_period_idx) * 0.8)
selection_train_idx = train_period_idx[:inner_cutoff]
selection_val_idx = train_period_idx[inner_cutoff:]
assert train_df_merged.loc[train_period_idx, "TransactionDT"].max() <= train_df_merged.loc[final_holdout_idx, "TransactionDT"].min()

X_id_prepared = train_df_merged.loc[train_period_idx, id_features_all].copy()
# Decide which indicators exist using the earlier training period only.
missing_indicator_features = []
for col in id_features_all:
    if X_id_prepared[col].isna().any():
        missing_col = f"{col}_is_missing"
        X_id_prepared[missing_col] = X_id_prepared[col].isna().astype("int8")
        missing_indicator_features.append(missing_col)
id_categorical_features = [col for col in id_features_all if (pd.api.types.is_object_dtype(X_id_prepared[col]) or pd.api.types.is_string_dtype(X_id_prepared[col]) or isinstance(X_id_prepared[col].dtype, pd.CategoricalDtype))]

def prepare_id_categories(frame, reference=None):
    frame = frame.copy()
    for col in id_categorical_features:
        values = frame[col].astype("object").fillna("__MISSING__")
        if reference is None:
            frame[col] = values.astype("category")
        else:
            frame[col] = pd.Categorical(values, categories=reference[col].cat.categories)
    return frame

X_train_id = prepare_id_categories(X_id_prepared.loc[selection_train_idx])
X_val_id = prepare_id_categories(X_id_prepared.loc[selection_val_idx], X_train_id)
y_train_id, y_val_id = y.loc[selection_train_idx], y.loc[selection_val_idx]
scale_pos_weight_value = (len(y_train_id) - y_train_id.sum()) / y_train_id.sum()
model = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
    learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, scale_pos_weight=scale_pos_weight_value,
    importance_type="gain", enable_categorical=True, tree_method="hist"
)
model.fit(X_train_id, y_train_id)
y_val_proba = model.predict_proba(X_val_id)[:, 1]
print("Inner validation ROC-AUC:", roc_auc_score(y_val_id, y_val_proba))
print("Inner validation PR-AUC:", average_precision_score(y_val_id, y_val_proba))
importance_df = pd.DataFrame({"feature": X_train_id.columns, "importance_gain": model.feature_importances_}).sort_values("importance_gain", ascending=False, kind="stable")
importance_df.head(30)


Inner validation ROC-AUC: 0.6783117442802457
Inner validation PR-AUC: 0.1834021399284021


,feature,importance_gain
34,id_35,0.370860
16,id_17,0.059456
9,id_10,0.059272
30,id_31,0.058196
0,id_01,0.034433
28,id_29,0.033533
8,id_09,0.032909
37,id_38,0.028482
32,id_33,0.023215
27,id_28,0.022105


In [8]:
X_missing_only = X_id_prepared[missing_indicator_features].copy()
X_train_miss = X_missing_only.loc[selection_train_idx]
X_val_miss = X_missing_only.loc[selection_val_idx]
y_train_miss, y_val_miss = y.loc[selection_train_idx], y.loc[selection_val_idx]
missing_model = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
    learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, scale_pos_weight=scale_pos_weight_value,
    importance_type="gain", tree_method="hist"
)
missing_model.fit(X_train_miss, y_train_miss)
y_val_miss_proba = missing_model.predict_proba(X_val_miss)[:, 1]
print("Inner validation missingness-only ROC-AUC:", roc_auc_score(y_val_miss, y_val_miss_proba))
print("Inner validation missingness-only PR-AUC:", average_precision_score(y_val_miss, y_val_miss_proba))
missing_only_importance_df = pd.DataFrame({"feature": X_train_miss.columns, "importance_gain": missing_model.feature_importances_}).sort_values("importance_gain", ascending=False, kind="stable")
missing_only_importance_df.head(30)


Inner validation missingness-only ROC-AUC: 0.6966324968216877
Inner validation missingness-only PR-AUC: 0.12145582234329608


,feature,importance_gain
1,id_02_is_missing,0.705206
12,id_13_is_missing,0.055920
8,id_09_is_missing,0.052920
9,id_10_is_missing,0.047029
13,id_14_is_missing,0.040558
31,id_32_is_missing,0.014621
17,id_18_is_missing,0.013240
29,id_30_is_missing,0.011763
33,id_34_is_missing,0.006406
3,id_04_is_missing,0.005648


> Identity features are predictive, but most of the predictive power comes from the values themselves, not just whether the values are missing.

Identity feature values and missingness are evaluated only on an inner validation slice within the earlier 80% training period. Rerun the cells above to obtain current scores. No final-holdout labels are used for feature ranking.

In [9]:
value_vs_missing = []

for col in id_features_all:
    value_importance = importance_df.loc[
        importance_df["feature"] == col,
        "importance_gain"
    ]

    missing_importance = importance_df.loc[
        importance_df["feature"] == f"{col}_is_missing",
        "importance_gain"
    ]

    value_vs_missing.append({
        "feature": col,
        "value_importance": value_importance.iloc[0] if len(value_importance) > 0 else 0,
        "missingness_importance": missing_importance.iloc[0] if len(missing_importance) > 0 else 0,
    })

value_vs_missing_df = pd.DataFrame(value_vs_missing)

value_vs_missing_df["total_importance"] = (
    value_vs_missing_df["value_importance"]
    + value_vs_missing_df["missingness_importance"]
)

value_vs_missing_df["missingness_share"] = (
    value_vs_missing_df["missingness_importance"]
    / value_vs_missing_df["total_importance"].replace(0, pd.NA)
)

value_vs_missing_df = value_vs_missing_df.sort_values(
    "total_importance",
    ascending=False
)

value_vs_missing_df.head(30)

,feature,value_importance,missingness_importance,total_importance,missingness_share
34,id_35,0.370860,0.000000e+00,0.370860,0.0
30,id_31,0.058196,7.242148e-03,0.065438,0.110672
16,id_17,0.059456,0.000000e+00,0.059456,0.0
9,id_10,0.059272,0.000000e+00,0.059272,0.0
0,id_01,0.034433,0.000000e+00,0.034433,0.0
28,id_29,0.033533,0.000000e+00,0.033533,0.0
8,id_09,0.032909,0.000000e+00,0.032909,0.0
37,id_38,0.028482,0.000000e+00,0.028482,0.0
32,id_33,0.023215,3.887518e-03,0.027103,0.143435
29,id_30,0.021792,4.927856e-03,0.026720,0.184428


In [10]:
# Top-k diagnostic on the inner validation period only.
X_train, X_val, y_train, y_val = X_train_id, X_val_id, y_train_id, y_val_id
ordered_features = importance_df["feature"].tolist()
cutoffs = [k for k in [1, 3, 5, 10, 15, 20, 30, 40, 50, len(ordered_features)] if k <= len(ordered_features)]
results = []
for k in cutoffs:
    top_k_features = ordered_features[:k]
    model_k = xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
        learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, scale_pos_weight=scale_pos_weight_value,
        enable_categorical=True, tree_method="hist"
    )
    model_k.fit(X_train[top_k_features], y_train)
    y_val_proba = model_k.predict_proba(X_val[top_k_features])[:, 1]
    results.append({"k": k, "pr_auc": average_precision_score(y_val, y_val_proba),
                    "roc_auc": roc_auc_score(y_val, y_val_proba), "features": top_k_features})
top_k_results_df = pd.DataFrame(results)
top_k_results_df[["k", "pr_auc", "roc_auc"]]


,k,pr_auc,roc_auc
0,1,0.091117,0.691959
1,3,0.118830,0.694955
2,5,0.160541,0.685691
3,10,0.175403,0.688481
4,15,0.181126,0.679949
5,20,0.190645,0.682821
6,30,0.186873,0.679719
7,40,0.185321,0.677426
8,50,0.181958,0.678152
9,76,0.182958,0.679014


The top-k sweep is an exploratory inner-validation diagnostic. It does not choose the exported feature count; the final list uses a fixed 95% cumulative gain rule fitted only on the earlier 80% training period.

The earlier top-20 list came from a random split across the full dataset and could include information from the final time holdout. Do not reuse that file. Generate the new list in the cell below.

In [11]:
import json
import numpy as np

# Refit the ID ranker on all earlier-period rows, with category levels learned there only.
X_selection_id = prepare_id_categories(X_id_prepared)
y_selection_id = y
selection_weight = (len(y_selection_id) - y_selection_id.sum()) / y_selection_id.sum()
final_id_ranker = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
    learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, scale_pos_weight=selection_weight,
    importance_type="gain", enable_categorical=True, tree_method="hist"
)
final_id_ranker.fit(X_selection_id, y_selection_id)
final_id_importance = pd.Series(final_id_ranker.feature_importances_, index=X_selection_id.columns).sort_values(ascending=False, kind="stable")
if final_id_importance.sum() <= 0:
    raise ValueError("No positive ID-feature gain")
selected_count = int(np.searchsorted((final_id_importance.cumsum() / final_id_importance.sum()).to_numpy(), 0.95, side="left")) + 1
selected_id_features = final_id_importance.head(selected_count).index.tolist()
import hashlib
feature_sha256 = hashlib.sha256(json.dumps(selected_id_features, separators=(",", ":")).encode()).hexdigest()
# Keep the list produced by this training-period fit. Record its exact digest
# so modeling notebooks use the same list, even if another platform ranks ties differently.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

transaction_csv_sha256 = sha256_file(TRAIN_TRANSACTION)
identity_csv_sha256 = sha256_file(TRAIN_IDENTITY)
selection_metadata = {
    "method": "XGBoost gain importance on chronological training rows only",
    "xgboost_version": xgb.__version__,
    "pandas_version": pd.__version__,
    "feature_sha256": feature_sha256,
    "transaction_csv_sha256": transaction_csv_sha256,
    "identity_csv_sha256": identity_csv_sha256,
    "gain_fraction": 0.95,
    "n_selection_rows": len(train_period_idx),
    "selection_time_max": float(train_df_merged.loc[train_period_idx, "TransactionDT"].max()),
    "holdout_time_min": float(train_df_merged.loc[final_holdout_idx, "TransactionDT"].min()),
    "n_selected_features": len(selected_id_features),
}
print("Selected", len(selected_id_features), "ID features from training-period gain only")


Selected 28 ID features from training-period gain only


In [12]:
selected_id_features_df = final_id_importance.loc[selected_id_features].rename("importance_gain").to_frame()
selected_id_features_df


,importance_gain
id_35,0.564574
id_17,0.043025
id_10,0.042249
id_31,0.040121
id_01,0.021605
id_09,0.018860
id_28,0.017973
id_29,0.016944
id_33,0.015139
id_30,0.014337


In [13]:
with open("selected_id_features.json", "w") as f:
    json.dump(selected_id_features, f, indent=2)
with open("id_feature_selection_metadata.json", "w") as f:
    json.dump(selection_metadata, f, indent=2)
print("Saved selected_id_features.json and id_feature_selection_metadata.json")


Saved selected_id_features.json and id_feature_selection_metadata.json


In [14]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
destination_dir = Path("/content/drive/My Drive/ML/Fraud Detection/Data/Artifacts")
destination_dir.mkdir(parents=True, exist_ok=True)
for filename in ["selected_id_features.json", "id_feature_selection_metadata.json"]:
    shutil.copy(filename, destination_dir / filename)
    print("Saved", destination_dir / filename)


Mounted at /content/drive
Saved /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/selected_id_features.json
Saved /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/id_feature_selection_metadata.json


# ID feature-selection findings

The exported list is now ranked by XGBoost gain fitted only on the earlier 80% of transactions. Its length is determined by the 95% cumulative-gain rule, so rerun the notebook to see the current count and selected columns. The inner validation diagnostics above can help inspect missingness and ranking behavior; their scores are not final holdout estimates.

# Overall EDA summary

The training-period exploration found fraud imbalance, useful identity values, and missingness signal. Rerun the inner-period diagnostics above for current scores. The exported V and ID feature lists use training-period gain importance and a 95% cumulative-gain rule; their counts are determined by the data, not fixed at 75 or 20.

# Modeling plan

1. Keep the latest 20% of transactions as a final time holdout.
2. Fit supervised feature selectors and preprocessing only on earlier-period rows.
3. Use an inner time split inside the earlier period for model and threshold tuning.
4. Compare models on the final holdout after choices are fixed.
5. Report PR-AUC, ROC-AUC, precision, recall, and threshold tradeoffs.

| Finding | Modeling decision |
|---|---|
| Missingness is target-associated | Add missing indicators or preserve missing values as an explicit category. |
| ProductCD fraud rates differ | Encode ProductCD as a categorical feature. |
| DeviceInfo is high-cardinality | Group rare values before encoding. |
| TransactionAmt is skewed | Test `log1p(TransactionAmt)` for linear models. |
| card/addr numeric columns act categorical | Avoid treating all numeric columns as continuous measurements. |
| Dataset is imbalanced | Use PR-AUC, recall, and threshold tuning instead of relying on accuracy. |
| Anonymized C/D/M features show potential signal | Inspect systematically and evaluate them during modeling instead of dropping them only because they lack semantic meaning. |
| Missingness in identity features is informative, but the actual identity feature values carry stronger predictive signal than missingness alone. | Use top 20 id features, exported in json file. |
| TransactionDT suggests that transactions are in a time period. | Use time based split instead of a random train test split. |

This notebook reserves the latest 20% by `TransactionDT`. The exploratory XGBoost fits and top-k diagnostics use only an inner time split within the earlier 80%. The exported ID list is refitted on that earlier 80%; no final-holdout rows or labels enter its ranking.